# Demo Backtest Viewer

Visualize outputs from `tools/run_backtest.py`, including active management diagnostics:
- Equity/NAV and drawdown
- Portfolio vs benchmark returns
- Turnover and cost
- Fundamental Law metrics (IC, Breadth, TC, implied vs realized IR)
- Horizon diagnostics (IC/spread/hit for h1/h2/h4)
- Weights and orders

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')


def _resolve_backtest_root() -> Path:
    candidates = [
        Path('eval_results/backtest'),
        Path('../eval_results/backtest'),
    ]
    for c in candidates:
        if c.exists():
            return c
    return candidates[0]


def _is_backtest_run_dir(path: Path) -> bool:
    return path.is_dir() and (path / 'summary.json').exists() and (path / 'equity_curve.csv').exists()


def _iter_valid_pairs(pair_root: Path) -> list[Path]:
    out: list[Path] = []
    if not pair_root.exists():
        return out
    pair_dirs = sorted(
        [d for d in pair_root.iterdir() if d.is_dir() and (d / 'comparison.json').exists()],
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    for d in pair_dirs:
        try:
            cmp_payload = json.loads((d / 'comparison.json').read_text())
            a_dir = Path(cmp_payload.get('run_a', {}).get('dir', d / 'run_a'))
            b_dir = Path(cmp_payload.get('run_b', {}).get('dir', d / 'run_b'))
            if _is_backtest_run_dir(a_dir) and _is_backtest_run_dir(b_dir):
                out.append(d)
        except Exception:
            continue
    return out



In [ ]:
BACKTEST_ROOT = _resolve_backtest_root()
all_runs = sorted(
    [p for p in BACKTEST_ROOT.iterdir() if _is_backtest_run_dir(p)],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
assert all_runs, f'No valid backtest runs found under: {BACKTEST_ROOT.resolve()}'

top3 = all_runs[:3]
recent_df = pd.DataFrame(
    [
        {
            'rank': i + 1,
            'run_dir': p.name,
            'modified_at': pd.to_datetime(p.stat().st_mtime, unit='s')
        }
        for i, p in enumerate(top3)
    ]
)
print('Top 3 recent backtest runs:')
display(recent_df)

# Defaults to most recent run. Uncomment to pin a specific run.
BACKTEST_DIR = top3[0]
# BACKTEST_DIR = BACKTEST_ROOT / 'demo_backtest_sp500_snapshot_ic_stable'
assert BACKTEST_DIR.exists(), f'Backtest dir not found: {BACKTEST_DIR.resolve()}'
print(f'Loading: {BACKTEST_DIR}')

summary = json.loads((BACKTEST_DIR / 'summary.json').read_text())
equity = pd.read_csv(BACKTEST_DIR / 'equity_curve.csv')
weights = pd.read_csv(BACKTEST_DIR / 'weights_history.csv')
orders = pd.read_csv(BACKTEST_DIR / 'orders_history.csv')

equity['trade_date'] = pd.to_datetime(equity['trade_date'])
equity['next_date'] = pd.to_datetime(equity['next_date'])
weights['trade_date'] = pd.to_datetime(weights['trade_date'])
if 'trade_date' in orders.columns:
    orders['trade_date'] = pd.to_datetime(orders['trade_date'])

summary



In [ ]:
summary_df = pd.DataFrame([summary]).T
summary_df.columns = ['value']
summary_df

## A/B Comparison (Latest 2 Runs)
Automatically compares the two most recent backtest runs under `eval_results/backtest`.


In [ ]:
PAIR_ROOT = BACKTEST_ROOT / 'ab_pairs'
pair_dirs = _iter_valid_pairs(PAIR_ROOT)

if pair_dirs:
    top_pairs = pair_dirs[:3]
    pair_df = pd.DataFrame(
        [
            {
                'rank': i + 1,
                'pair_dir': d.name,
                'modified_at': pd.to_datetime(d.stat().st_mtime, unit='s'),
            }
            for i, d in enumerate(top_pairs)
        ]
    )
    print('Top 3 recent A/B pairs:')
    display(pair_df)

    pair_dir = top_pairs[0]
    cmp_payload = json.loads((pair_dir / 'comparison.json').read_text())

    a_dir = Path(cmp_payload.get('run_a', {}).get('dir', pair_dir / 'run_a'))
    b_dir = Path(cmp_payload.get('run_b', {}).get('dir', pair_dir / 'run_b'))
    labels = [
        cmp_payload.get('labels', {}).get('a', 'A'),
        cmp_payload.get('labels', {}).get('b', 'B'),
    ]
    run_dirs = [a_dir, b_dir]

    summaries = []
    eq_map = {}
    for label, run_dir in zip(labels, run_dirs):
        s = json.loads((run_dir / 'summary.json').read_text())
        summaries.append({
            'label': label,
            'dir': run_dir.name,
            'total_return': s.get('total_return'),
            'sharpe': s.get('sharpe'),
            'max_drawdown': s.get('max_drawdown'),
            'tracking_error': s.get('tracking_error'),
            'realized_active_ir': s.get('realized_active_information_ratio'),
            'implied_ir': s.get('implied_information_ratio'),
            'average_ic': s.get('average_ic'),
            'avg_transfer_coefficient': s.get('average_transfer_coefficient_proxy'),
            'avg_executed_turnover': s.get('average_executed_turnover'),
        })
        eq = pd.read_csv(run_dir / 'equity_curve.csv')
        eq['trade_date'] = pd.to_datetime(eq['trade_date'])
        eq_map[label] = eq

    cmp_df = pd.DataFrame(summaries).set_index('label')
    display(cmp_df)

    if len(labels) == 2:
        numeric_cols = cmp_df.select_dtypes(include='number').columns.tolist()
        delta = (cmp_df.loc[labels[1], numeric_cols] - cmp_df.loc[labels[0], numeric_cols]).to_frame(name=f'{labels[1]}_minus_{labels[0]}').T
        display(delta)

    fig, ax = plt.subplots(figsize=(10, 4))
    for label, run_dir in zip(labels, run_dirs):
        eq = eq_map[label].sort_values('trade_date').copy()
        norm_nav = eq['nav'] / eq['nav'].iloc[0]
        ax.plot(eq['trade_date'], norm_nav, label=f"{label} ({run_dir.name})")
    ax.set_title(f"A/B Pair Normalized NAV ({pair_dir.name})")
    ax.set_xlabel('trade_date')
    ax.set_ylabel('normalized nav')
    ax.legend()
    plt.tight_layout()
    plt.show()

else:
    ab_runs = all_runs[:2]
    if len(ab_runs) < 2:
        print('Need at least 2 valid runs for A/B comparison under', BACKTEST_ROOT.resolve())
    else:
        labels = ['A_latest', 'B_previous']
        summaries = []
        eq_map = {}
        for label, run_dir in zip(labels, ab_runs):
            s = json.loads((run_dir / 'summary.json').read_text())
            summaries.append({
                'label': label,
                'dir': run_dir.name,
                'total_return': s.get('total_return'),
                'sharpe': s.get('sharpe'),
                'max_drawdown': s.get('max_drawdown'),
                'tracking_error': s.get('tracking_error'),
                'realized_active_ir': s.get('realized_active_information_ratio'),
                'implied_ir': s.get('implied_information_ratio'),
                'average_ic': s.get('average_ic'),
                'avg_transfer_coefficient': s.get('average_transfer_coefficient_proxy'),
                'avg_executed_turnover': s.get('average_executed_turnover'),
            })
            eq = pd.read_csv(run_dir / 'equity_curve.csv')
            eq['trade_date'] = pd.to_datetime(eq['trade_date'])
            eq_map[label] = eq

        cmp_df = pd.DataFrame(summaries).set_index('label')
        display(cmp_df)

        numeric_cols = cmp_df.select_dtypes(include='number').columns.tolist()
        delta = (cmp_df.loc['A_latest', numeric_cols] - cmp_df.loc['B_previous', numeric_cols]).to_frame(name='A_minus_B').T
        display(delta)

        fig, ax = plt.subplots(figsize=(10, 4))
        for label in labels:
            eq = eq_map[label].sort_values('trade_date').copy()
            norm_nav = eq['nav'] / eq['nav'].iloc[0]
            ax.plot(eq['trade_date'], norm_nav, label=f"{label} ({ab_runs[labels.index(label)].name})")
        ax.set_title('A/B Normalized NAV (Latest 2 Runs)')
        ax.set_xlabel('trade_date')
        ax.set_ylabel('normalized nav')
        ax.legend()
        plt.tight_layout()
        plt.show()



## Equity and Drawdown

In [ ]:
eq_plot = equity.sort_values('trade_date').copy()
initial_capital = float(summary.get('initial_capital', eq_plot['nav'].iloc[0]))
eq_plot['benchmark_nav'] = initial_capital * (1.0 + eq_plot['benchmark_return'].fillna(0.0)).cumprod()

fig, ax = plt.subplots(figsize=(10, 4))
eq_plot.plot(x='trade_date', y='nav', ax=ax, label='portfolio_nav', alpha=0.9)
eq_plot.plot(x='trade_date', y='benchmark_nav', ax=ax, label='benchmark_nav_ref', alpha=0.9)
ax.set_title('Equity Curve (Portfolio vs Benchmark Reference NAV)')
ax.set_xlabel('trade_date')
ax.set_ylabel('nav')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
eq = equity[['trade_date', 'nav']].copy()
eq['running_max'] = eq['nav'].cummax()
eq['drawdown'] = eq['nav'] / eq['running_max'] - 1.0

fig, ax = plt.subplots(figsize=(10, 3))
eq.plot(x='trade_date', y='drawdown', ax=ax, legend=False, title='Drawdown')
ax.set_xlabel('trade_date')
ax.set_ylabel('drawdown')
plt.tight_layout()
plt.show()

eq[['trade_date', 'drawdown']].sort_values('drawdown').head(5)

## Returns and Turnover

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
equity.plot(x='trade_date', y='portfolio_return', ax=ax, label='portfolio_return', alpha=0.85)
equity.plot(x='trade_date', y='benchmark_return', ax=ax, label='benchmark_return', alpha=0.85)
ax.set_title('Periodic Returns')
ax.set_xlabel('trade_date')
ax.set_ylabel('return')
plt.tight_layout()
plt.show()

In [ ]:
turn_cols = [c for c in ['raw_turnover', 'executed_turnover', 'turnover_constraint_drag'] if c in equity.columns]
fig, ax = plt.subplots(figsize=(10, 4))
if turn_cols:
    equity.plot(x='trade_date', y=turn_cols, ax=ax, title='Turnover Decomposition')
else:
    equity.plot(x='trade_date', y='turnover', ax=ax, legend=False, title='Turnover by Rebalance')
ax.set_xlabel('trade_date')
ax.set_ylabel('turnover')
plt.tight_layout()
plt.show()

view_cols = [c for c in ['trade_date', 'turnover', 'raw_turnover', 'executed_turnover', 'turnover_constraint_drag', 'cost'] if c in equity.columns]
equity[view_cols].tail(10)

## Fundamental Law Diagnostics

In [ ]:
law_keys = [
    'average_ic',
    'average_breadth_proxy',
    'average_transfer_coefficient_proxy',
    'implied_information_ratio',
    'realized_active_information_ratio',
]
law = {k: summary.get(k) for k in law_keys}
pd.DataFrame([law]).T.rename(columns={0: 'value'})

In [ ]:
ir_df = pd.DataFrame({
    'type': ['implied_ir', 'realized_active_ir'],
    'value': [summary.get('implied_information_ratio'), summary.get('realized_active_information_ratio')]
})
fig, ax = plt.subplots(figsize=(6, 3))
ir_df.plot(kind='bar', x='type', y='value', ax=ax, legend=False, title='Implied vs Realized Active IR')
ax.set_ylabel('IR')
plt.tight_layout()
plt.show()
ir_df

## Horizon Diagnostics (h1/h2/h4)

In [ ]:
horizon_cols = [c for c in equity.columns if c.startswith('metric_ic_h') or c.startswith('metric_spread_h') or c.startswith('metric_hit_h')]
horizon_cols[:20], len(horizon_cols)

In [ ]:
ic_cols = [c for c in equity.columns if c.startswith('metric_ic_h')]
if ic_cols:
    fig, ax = plt.subplots(figsize=(10, 4))
    equity.plot(x='trade_date', y=ic_cols, ax=ax, title='Horizon IC Time Series')
    ax.set_xlabel('trade_date')
    ax.set_ylabel('IC')
    plt.tight_layout()
    plt.show()

    ic_summary = pd.DataFrame({
        'horizon': ic_cols,
        'mean_ic': [equity[c].mean() for c in ic_cols],
        'std_ic': [equity[c].std(ddof=0) for c in ic_cols],
    })
    ic_summary['ic_ir_proxy'] = ic_summary['mean_ic'] / ic_summary['std_ic']
    ic_summary
else:
    print('No horizon IC columns found.')

In [ ]:
spread_cols = [c for c in equity.columns if c.startswith('metric_spread_h')]
hit_cols = [c for c in equity.columns if c.startswith('metric_hit_h')]

if spread_cols:
    fig, ax = plt.subplots(figsize=(10, 4))
    equity.plot(x='trade_date', y=spread_cols, ax=ax, title='Top-Bottom Spread by Horizon')
    ax.set_xlabel('trade_date')
    ax.set_ylabel('spread')
    plt.tight_layout()
    plt.show()

if hit_cols:
    fig, ax = plt.subplots(figsize=(10, 3))
    equity.plot(x='trade_date', y=hit_cols, ax=ax, title='Top-Bottom Hit Rate Signal (0/1)')
    ax.set_xlabel('trade_date')
    ax.set_ylabel('hit')
    plt.tight_layout()
    plt.show()

horizon_rollup = {}
for c in spread_cols + hit_cols:
    horizon_rollup[c] = float(equity[c].mean())
pd.DataFrame([horizon_rollup]).T.rename(columns={0: 'average'})

## Weights and Orders

In [ ]:
weight_cols = [c for c in weights.columns if c != 'trade_date']
avg_weights = weights[weight_cols].mean().sort_values(ascending=False)
avg_weights.head(20)

In [ ]:
top_avg = avg_weights.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
top_avg.plot(kind='barh', ax=ax, title='Top 20 Average Weights')
ax.set_xlabel('average weight')
plt.tight_layout()
plt.show()

In [ ]:
orders.head(20)

In [ ]:
order_stats = {
    'total_orders': int(len(orders)),
    'total_estimated_cost': float(orders['estimated_cost'].sum()) if 'estimated_cost' in orders.columns else None,
    'buy_orders': int((orders['action'] == 'BUY').sum()) if 'action' in orders.columns else None,
    'sell_orders': int((orders['action'] == 'SELL').sum()) if 'action' in orders.columns else None,
}
order_stats